# **랜덤포레스트 모델 돌리기 전 전처리**
### 251012 1에서 5차 통합여성관리자 데이터 결측치 채우기 작업

In [33]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_squared_error


ModuleNotFoundError: No module named 'sklearn'

In [67]:
# 파일 열기
filename = "../../data/jooui/data_0804_long.csv"
filename_filtered = filename.replace(".csv", "_filtered_251012.csv")
SHEET = "sheet"

df = pd.read_csv(filename, encoding='utf-8', header=1,encoding_errors='ignore')

key_df = df.keys()

C:\Users\0525s\AppData\Local\Temp\ipykernel_3244\4198153221.py:6: DtypeWarning: Columns (20,39,58,77,96,109,191,196,201,206,211,217,222,391,439,449,458,467,472,475,661,701,704,722,738,754,777,811,898,900,902,904,908,912,918,920,926,931,950) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename, encoding='utf-8', header=1,encoding_errors='ignore')


In [68]:
# 인코딩은 상황에 맞게 (utf-8, utf-8-sig, cp949 중 선택)
df0 = pd.read_csv(filename, header=None, nrows=2, encoding='cp949')

# 0행: 값 / 1행: 키
val_row = df0.iloc[0]
key_row = df0.iloc[1]

# zip으로 묶어서 dict 생성
data_dict = dict(zip(key_row, val_row))

for idx, k in enumerate(data_dict.keys()):
    if idx<10:
        print(f"{k} : {data_dict[k]}")

ID : [기본정보] ID
WAVE : [기본정보] 응답 차수
METHOD : [조사방법] 조사방법
LISTID : [기본정보] 사업체 코드
SQ0 : [기본정보] 조직형태(인사담당자)
SQ0_1 : [기본정보] 설립년도(인사담당자)
AREA : [기본정보] 지역
AREA2 : [기본정보] 지역(가공변수)
AJ : [기본정보] 산업대분류
AJ2 : [기본정보] 산업대분류(가공변수)


In [69]:
df.head(20)

,ID,WAVE,METHOD,LISTID,SQ0,SQ0_1,AREA,AREA2,AJ,AJ2,...,DP_C8_5,DP_C8_6,DP_C8_7,DP_C8_8,DP_C9,DP_C9_1_M1,DP_C9_1_M2,DP_C9_1_OP,P_WGT_FF,P_WGT_AF
0,1,1,1,1.0,4.0,2005.0,11.0,5.0,18.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.452305,NaN
1,2,1,1,1.0,4.0,2005.0,11.0,5.0,18.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.649474,NaN
2,3,1,1,1.0,4.0,2005.0,11.0,5.0,18.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.649474,NaN
3,4,1,1,1.0,4.0,2005.0,11.0,5.0,18.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.649474,NaN
4,5,1,1,40.0,1.0,2009.0,2.0,3.0,17.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.269288,NaN
5,6,1,1,40.0,1.0,2009.0,2.0,3.0,17.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.269288,NaN
6,7,1,1,40.0,1.0,2009.0,2.0,3.0,17.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.269288,NaN
7,8,1,1,40.0,1.0,2009.0,2.0,3.0,17.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.269288,NaN
8,9,1,1,40.0,1.0,2009.0,2.0,3.0,17.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.269288,NaN
9,10,1,1,56.0,1.0,2003.0,6.0,5.0,14.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.500284,NaN


In [70]:
df.keys()

Index(['ID', 'WAVE', 'METHOD', 'LISTID', 'SQ0', 'SQ0_1', 'AREA', 'AREA2', 'AJ',
       'AJ2',
       ...
       'DP_C8_5', 'DP_C8_6', 'DP_C8_7', 'DP_C8_8', 'DP_C9', 'DP_C9_1_M1',
       'DP_C9_1_M2', 'DP_C9_1_OP', 'P_WGT_FF', 'P_WGT_AF'],
      dtype='object', length=953)

### **NaN 값들 처리** => 84개 데이터 정도면 랜덤포레스트하기 충분한가?

In [71]:
# NaN 비율 확인 => 지울 데이터 정하기 위함
# 비율별 NaN 갯수 확인하고, 남길 데이터의 갯수 확인
nan_ratio = df.isna().mean() * 100
print(nan_ratio)


ID             0.000000
WAVE           0.000000
METHOD         0.000000
LISTID         2.433177
SQ0            2.433177
                ...    
DP_C9_1_M1    99.122628
DP_C9_1_M2    99.122628
DP_C9_1_OP    99.943889
P_WGT_FF       7.896348
P_WGT_AF      46.893491
Length: 953, dtype: float64


In [72]:
import pandas as pd

# NaN 비율과 NaN 개수를 계산
nan_count = df.isna().sum()
nan_ratio = df.isna().mean() * 100

# 두 정보를 하나의 DataFrame으로 합치기
nan_summary = pd.DataFrame({
    'NaN 개수': nan_count,
    'NaN 비율(%)': nan_ratio.round(2)
}).sort_values('NaN 비율(%)', ascending=False)

# 보기 좋게 출력
# print("**열별 NaN 통계 요약**")
# print(nan_summary.head(20))

print(len(nan_summary))
nan_summary[-200:-170]  # 확인해본 결과 약 84개의 열이 NaN 비율이 10%대 이하임

953


,NaN 개수,NaN 비율(%)
B4_5,12658,64.57
B4_1,12658,64.57
B4_2,12658,64.57
B4_3,12658,64.57
B4_4,12658,64.57
D5_1_1_N2,11428,58.29
D5_1_1_N1,11428,58.29
B9_2_2,11078,56.51
B9_11_2,10746,54.82
B9_1_2,10678,54.47


In [73]:
nan_summary[-86:-60]

,NaN 개수,NaN 비율(%)
F2,3533,18.02
D1_1_N1,3533,18.02
B9_12_3,2036,10.39
B9_12_2,2036,10.39
B9_4_2,1562,7.97
B9_4_3,1562,7.97
P_WGT_FF,1548,7.90
B9_5_3,1402,7.15
B9_5_2,1402,7.15
B9_8_1,477,2.43


In [74]:
col_names = nan_summary.index.tolist()
print(col_names[-82:])  # 앞부분만 확인

['B9_4_2', 'B9_4_3', 'P_WGT_FF', 'B9_5_3', 'B9_5_2', 'B9_8_1', 'B9_9_1', 'B9_7_1', 'F14_3', 'B9_6_1', 'AJ2', 'LISTID', 'SQ0', 'SQ0_1', 'AREA', 'AREA2', 'AJ', 'TOTAL_SIZE', 'B9_5_1', 'SQ4', 'B6_N1', 'B6_N2', 'B9_1_1', 'B9_2_1', 'B9_4_1', 'B6_N3', 'C5_2', 'B9_10_1', 'F11_8', 'E5_4_1', 'E5_5_1', 'E5_6_1', 'E5_7_1', 'E5_8_1', 'F9', 'F10', 'F11_5', 'F11_6', 'F11_7', 'F12_1', 'E5_2_1', 'F12_2', 'F12_3', 'F12_4', 'F12_5', 'F13_1', 'F13_2', 'F13_3', 'F13_4', 'F13_5', 'F14_1', 'B9_11_1', 'E5_3_1', 'E5_1_1', 'D2_3', 'B9_12_1', 'B9_13_1', 'B9_14_1', 'C5_1', 'F14_2', 'C5_3', 'C5_4', 'C5_5', 'D1', 'D2_2', 'D2_1', 'D2_4', 'D3', 'D4_1', 'D4_2', 'D4_3', 'D4_4', 'D4_5', 'E2_1_M1', 'E2_2_M1', 'E2_3_M1', 'WAVE', 'AGE', 'SQ2', 'SEX', 'METHOD', 'ID']


In [75]:
for idx, k in enumerate(col_names[-82:]):
    if idx<10:
        print(k)

B9_4_2
B9_4_3
P_WGT_FF
B9_5_3
B9_5_2
B9_8_1
B9_9_1
B9_7_1
F14_3
B9_6_1


In [76]:
# 실제 질문을 nan_summary 뒤에 열로 추가해줌
nan_summary["question"] = nan_summary.index.to_series().apply(lambda x: data_dict.get(x, "미정"))

# # 결측치 관련 내용들 파일로 저장
# nan_summary_filtered = nan_summary[-86:]
# nan_summary_filtered.to_csv("nan_summary.csv", encoding="utf-8-sig")

In [77]:
nan_summary[-86:-60]

,NaN 개수,NaN 비율(%),question
F2,3533,18.02,[가구상황] 배우자 유무
D1_1_N1,3533,18.02,[직장 내 관계 및 네트워크] 팀의 총 인원
B9_12_3,2036,10.39,[현 직장 사항] 직장 내 일생활균형 관련 제도 (12) 병가휴직 - 조직에서 쉽게...
B9_12_2,2036,10.39,[현 직장 사항] 직장 내 일생활균형 관련 제도 (12) 병가휴직 - 본인의 사용여부
B9_4_2,1562,7.97,[현 직장 사항] 직장 내 일생활균형 관련 제도 (4) 출산전후휴가(배우자출산휴가 ...
B9_4_3,1562,7.97,[현 직장 사항] 직장 내 일생활균형 관련 제도 (4) 출산전후휴가(배우자출산휴가 ...
P_WGT_FF,1548,7.90,균형 종단가중치
B9_5_3,1402,7.15,[현 직장 사항] 직장 내 일생활균형 관련 제도 (5) 육아휴직 - 조직에서 쉽게 ...
B9_5_2,1402,7.15,[현 직장 사항] 직장 내 일생활균형 관련 제도 (5) 육아휴직 - 본인의 사용여부
B9_8_1,477,2.43,[현 직장 사항] 직장 내 일생활균형 관련 제도 (8) 직장보육시설 - 현재 제도 유무


### **결측값을 -9로 채움**

In [78]:
# 결측치 -9로 채우기
# 1️⃣ -9 값이 존재하는지 확인
has_minus9 = (df == -9).any().any()   # DataFrame 전체에서 -9가 있는지 (True / False)

# 2️⃣ 없으면 결측치(NaN)를 -9로 대체
if not has_minus9:
    df = df.fillna(-9)
    print("✅ -9 값이 없어서, 모든 NaN을 -9로 대체했습니다.")
else:
    print("⚠️ 이미 -9 값이 존재합니다. 결측치 변경은 수행하지 않았습니다.")

✅ -9 값이 없어서, 모든 NaN을 -9로 대체했습니다.


In [79]:
df.to_csv(filename_filtered, encoding="utf-8-sig")